### Using Nemoguardrails

[NVIDIA's Nemoguardrails](https://docs.nvidia.com/nemo/guardrails/latest/home) lets you organize multiple guardrails at once and combine different models and approaches in a configuration approach.

The basics of the design allow you to define models to load, and then input and output guardrails to run. There are additional guardrails that can follow multi-turn and tool calls. You can use the configuration language to set up different configurations for different applications. I think especially helpful is that you can run them as a service, allowing you to treat it like a gateway.  

There is fairly active development, so always check the latest documentation to get an idea if something has changed...

In [ ]:
!cat nemoguardrails/config/config.yml

In [ ]:
!ollama list 

If you don't see llama-guard3:latest, you can update config with whatever llamaguard you have or run ``ollama pull llama-guard3:latest``

And now let's look at what prompts are being used.

In [ ]:
!cat nemoguardrails/config/prompts.yml

#### Do some tutorials to learn more...

There are many more examples [in the tutorials, although some require NVIDIA API KEYS](https://docs.nvidia.com/nemo/guardrails/get-started/tutorials)

NVIDIA also developed a special language called [Colang](https://docs.nvidia.com/nemo/guardrails/configure-guardrails/colang/) to help define guardrails. I used an LLM to help me format this by just telling it what version of Colang and having it mess up a few times before we got it correct... :) 


In [ ]:
!cat nemoguardrails/config/rails.co

In [ ]:
import phoenix as px
from phoenix.otel import register
from openinference.instrumentation.openai import OpenAIInstrumentor
from opentelemetry import trace
from openai import OpenAI

## Start your OTel Trace collection with Arize

If you haven't already installed it, set up [Arize Phoenix](https://arize.com/phoenix/). 

I think you can run it just with this command: ``uvx arize-phoenix serve``

If already installed, in another terminal, run:  ``phoenix serve``

In [ ]:
register(
    project_name="nemo-openai-client",
    endpoint="http://localhost:6006/v1/traces",
    set_global_tracer_provider=True
)
OpenAIInstrumentor().instrument()

tracer = trace.get_tracer("nemo-client")

### Getting nemoguardrails running

If not yet installed, run: ``uv pip install nemoguardrails``

Then make sure the terminal is in the nemoguardrails folder in this repository (so it can access the config files), and then run ``nemoguardrails server --config=./config --default-config-id=config --port 8000``

You should see it start up without error...


### Sending some data to our guardrails...

In [ ]:
client = OpenAI(
    base_url="http://localhost:8000/v1", # "http://<Your api-server IP>:port"
    api_key = "sk-no-key-required"
)

In [ ]:
response = client.chat.completions.create(
    model="llama3",
    messages=[{"role": "user", "content": "Hello"}],
    extra_body={"config_id": "config"} # this name should match if you change configuration names/serving
)

In [ ]:
response.__dict__

In [ ]:
response.choices

In [ ]:
response.messages

In [ ]:
response.log

In [ ]:
response.log.keys()

### You can now check it in Arize dashboard too! 

If running locally, go to your [Arize dashboard](http://127.0.0.1:6006/projects/). You should have a new project there...

### Your choice

1. You can navigate directly to your [Guardrails inteactive chat in a browser](http://127.0.0.1:8000) and start a chat there and see if you can trigger any guardrails.

2. Continue below with less design but some traces instead :) 

In [ ]:
def build_conversation(client, user_prompt, conversation):
    with tracer.start_as_current_span("nemo_execution") as span:
        conversation.append(
            {'role': 'user', 
             'content': user_prompt})
        response = client.chat.completions.create(
            model="llama3",
            messages=conversation, 
            extra_body={
                "config_id": "config"
            }
        )
        conversation.append(response.messages[-1])
    print(response.messages[-1].get('content'))
    return conversation

In [ ]:
system_prompt = """
You are a helpful chat assistant..."""

In [ ]:
def iterate_convo(client, system_prompt, stop_word="quit"):
    conversation = [
        {'role': 'system',
         'content': system_prompt},
    ]
    user_input = input(">")
    while user_input != stop_word:
        conversation = build_conversation(client, user_input, conversation)
        user_input = input(">")

In [ ]:
# type quit to exit at any time or just press the stop button.
iterate_convo(client, system_prompt)